In [1]:
!pip install -q ultralytics onnx onnxruntime opencv-python pillow

In [ ]:
import os
os.kill(os.getpid(), 9)

In [2]:
from ultralytics import YOLO

print("Ultralytics uğurla yükləndi")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics uğurla yükləndi


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

model_path = "/content/runs/detect/cats_v2_run1/weights/best.pt"

print(os.path.exists(model_path))

False


In [6]:
!find /content -name "best.pt"

/content/drive/MyDrive/cat_detection_copy/runs/cats_v1/weights/best.pt


In [8]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/cat_detection_copy/runs/cats_v1/weights/best.pt")

model.export(
    format="onnx",
    imgsz=640,
    opset=17,
    dynamic=False
)

Ultralytics 8.4.58 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26s summary (fused): 122 layers, 9,465,567 parameters, 0 gradients, 20.5 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/cat_detection_copy/runs/cats_v1/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (76.6 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 10 packages in 332ms
Prepared 2 packages in 48ms
Installed 2 packages in 9ms
 + colorama==0.4.6
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 1.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 17...


/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 17 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 5.3s, saved as '/content/drive/MyDrive/cat_detection_copy/runs/cats_v1/weights/best.onnx' (36.4 MB)

Export complete (7.1s)
Results saved to /content/drive/MyDrive/cat_detection_copy/runs/cats_v1/weights/best.onnx
Predict:         yolo predict task=detect model=/content/drive/MyDrive/cat_detection_copy/runs/cats_v1/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/drive/MyDrive/cat_detection_copy/runs/cats_v1/weights/best.onnx imgsz=640 data=/content/drive/MyDrive/cat_detection_copy/data.yaml  
Visualize:       https://netron.app


'/content/drive/MyDrive/cat_detection_copy/runs/cats_v1/weights/best.onnx'

In [9]:
import onnxruntime as ort

model_path = "/content/drive/MyDrive/cat_detection_copy/runs/cats_v1/weights/best.onnx"

session = ort.InferenceSession(model_path)

print("Model yükləndi")
print("Input:", session.get_inputs()[0].shape)
print("Output:", session.get_outputs()[0].shape)

Model yükləndi
Input: [1, 3, 640, 640]
Output: [1, 300, 6]


In [12]:
!ls /content/drive/MyDrive/cat_detection_copy

data.yaml  labels	 runs	  test.txt   val.txt
images	   labels.cache  _splits  train.txt


In [14]:
!find /content/drive/MyDrive/cat_detection_copy -name "*.jpg" | head -1

/content/drive/MyDrive/cat_detection_copy/images/4cb664f9bb4b05b1.jpg


In [16]:
img_path = "/content/drive/MyDrive/cat_detection_copy/images/4cb664f9bb4b05b1.jpg"

In [17]:
from ultralytics import YOLO

pt_model = YOLO(
    "/content/drive/MyDrive/cat_detection_copy/runs/cats_v1/weights/best.pt"
)

pt_result = pt_model(img_path)

print(pt_result[0].boxes.xyxy)


image 1/1 /content/drive/MyDrive/cat_detection_copy/images/4cb664f9bb4b05b1.jpg: 448x640 2 cats, 447.3ms
Speed: 17.2ms preprocess, 447.3ms inference, 3.0ms postprocess per image at shape (1, 3, 448, 640)
tensor([[   0.0000,  114.6877, 1737.7650, 1426.0968],
        [ 193.7053,   41.3977, 2065.6951, 1321.6757]])


In [18]:
onnx_model = YOLO(
    "/content/drive/MyDrive/cat_detection_copy/runs/cats_v1/weights/best.onnx"
)

onnx_result = onnx_model(img_path)

print(onnx_result[0].boxes.xyxy)

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Loading /content/drive/MyDrive/cat_detection_copy/runs/cats_v1/weights/best.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.26.0 with CPUExecutionProvider

image 1/1 /content/drive/MyDrive/cat_detection_copy/images/4cb664f9bb4b05b1.jpg: 640x640 2 cats, 1325.6ms
Speed: 7.7ms preprocess, 1325.6ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)
tensor([[   0.0000,  121.6408, 1774.5144, 1407.3936],
        [ 181.8849,   39.3442, 2053.2498, 1307.9170]])


### ONNX Validation

The exported ONNX model was loaded successfully using ONNX Runtime.

A test image containing two cats was evaluated with both the original PyTorch model and the exported ONNX model.

Results:

- PyTorch detections: 2 cats
- ONNX detections: 2 cats

The predicted bounding boxes were nearly identical, with only minor numerical differences expected from the export process and runtime implementation.

This confirms that the ONNX export preserved the detector behaviour and is suitable for deployment.